# Лекция 13. Генерация текстов

Существует два основных способа генерации текста. Можно анализировать данные и делать прогнозы относительно них на уровне **слов** в корпусе или на уровне отдельных **символов**. Как генерация на уровне символов, так и генерация на уровне слов имеют свои преимущества и недостатки.

Как правило, языковые модели на уровне слов имеют тенденцию демонстрировать более высокую точность, чем языковые модели на уровне символов. Это происходит потому, что они могут формировать более короткие представления предложений и сохранять контекст между словами легче, чем языковые модели на уровне символов. Однако для достаточной подготовки языковых моделей уровня слов необходимы большие корпуса, а однократное кодирование не очень осуществимо для моделей уровня слов.

Языковые модели на уровне символов быстрее обучаются, требуют меньше памяти и имеют более быстрый вывод, чем словесные модели. Это связано с тем, что “словарный запас” (количество обучающих функций) для модели будет намного меньше в целом, ограничен несколькими сотнями символов, а не сотнями тысяч слов.

Символьные модели также хорошо работают при переводе слов между языками, потому что они захватывают символы, которые составляют слова, а не пытаются захватить семантические качества слов.

Для обучения нейронной сети понадобится файл с текстом. Содержание данного файла будет определять тематику и содержание генерируемых текстов. На основе содержимого этого файла будет выполняться обучение нейронной сети.

В рамках данной практической работы мы построим самую простую рекуррентную нейросеть, которая не разбирается в языке, построении фраз, предложений, смыслах, а просто учится предсказывать следующий символ по предыдущему тексту.
Поскольку мы будем предсказывать только один символ, то нет необходимости учитывать правила пунктуации, выбирать заглавную или строчную букву. Нейронная сеть сможет это сделать самостоятельно на основании тех закономерностей, которые она подчерпнет из предложенного ей текста (если в тексте не будет заглавных букв, то их и не будет в генерируемом тексте).
Основным минусом такого подхода будет сильная зависимость нейронной сети от предложенного текста, она сможет генерировать фразы и соединять их друг с другом, но не всегда удачно. Поэтому мы всегда будем понимать, что этот текст был составлен именно нейронной сетью.

In [21]:
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

Токенизируем текст. Суть токенизации текста - превратить его в индексы. Т.е. находим в тексте все уникальные символы: буквы, пробелы, знаки препинания и каждому символу присваиваем число. Делаем прямой и обратный (индексы в символы) словарь. А потом проходимся по всему тексту и превращаем с помощью составленного нами словаря каждый символ в индекс.

В нашем случае используемый текст романа "Война и мир" Л.Н. Толстого. При сохранении текстового файла необходимо использовать кодировку "Кириллица Windows".

In [36]:
TRAIN_TEXT_FILE_PATH = 'Book.txt'

with open(TRAIN_TEXT_FILE_PATH) as text_file:
    text_sample = text_file.readlines()
text_sample = ' '.join(text_sample)

def text_to_seq(text_sample):
    char_counts = Counter(text_sample)
    char_counts = sorted(char_counts.items(), key = lambda x: x[1], reverse=True)

    sorted_chars = [char for char, _ in char_counts]
    print(sorted_chars)
    char_to_idx = {char: index for index, char in enumerate(sorted_chars)}
    idx_to_char = {v: k for k, v in char_to_idx.items()}
    sequence = np.array([char_to_idx[char] for char in text_sample])
    
    return sequence, char_to_idx, idx_to_char

sequence, char_to_idx, idx_to_char = text_to_seq(text_sample)

[' ', 'о', 'а', 'е', 'и', 'н', 'т', 'с', 'л', 'в', 'р', 'к', 'д', ',', 'м', 'у', 'п', 'я', 'ь', 'г', 'ы', 'з', 'б', 'ч', '\n', '.', 'й', 'ж', 'ш', 'х', '–', '\xa0', 'ю', 'e', 'ц', 'щ', 'Н', 'э', 's', 'n', 'r', 'i', 'a', 'П', '-', 'u', 'o', 't', 'А', 'В', 'ф', 'l', '3', ';', 'О', '!', '?', '2', 'К', 'Б', 'm', 'Д', 'М', 'd', 'c', 'С', 'Р', '&', '#', '…', 'И', 'p', 'Т', 'v', ':', 'Я', '»', '«', '’', '4', 'h', 'Г', '1', 'ъ', '[', ']', 'Э', 'Ч', 'q', 'Е', 'f', ')', '(', 'b', 'g', 'Л', '5', 'I', '8', '0', 'З', 'У', '6', '9', '7', 'z', 'j', 'Ж', 'V', 'M', 'B', 'A', 'X', 'Ф', 'Х', 'x', 'Ш', 'L', '\t', 'y', 'C', 'J', 'N', 'P', 'ё', 'k', 'E', 'S', 'D', 'Ц', 'R', 'w', 'T', 'Q', 'O', '/', 'G', 'F', 'H', 'U', 'K', 'Ю', '_', '*', 'W', '„', '“', 'Z', 'Щ', '<', '>', '—', 'Й']


In [37]:
print(char_to_idx)

{' ': 0, 'о': 1, 'а': 2, 'е': 3, 'и': 4, 'н': 5, 'т': 6, 'с': 7, 'л': 8, 'в': 9, 'р': 10, 'к': 11, 'д': 12, ',': 13, 'м': 14, 'у': 15, 'п': 16, 'я': 17, 'ь': 18, 'г': 19, 'ы': 20, 'з': 21, 'б': 22, 'ч': 23, '\n': 24, '.': 25, 'й': 26, 'ж': 27, 'ш': 28, 'х': 29, '–': 30, '\xa0': 31, 'ю': 32, 'e': 33, 'ц': 34, 'щ': 35, 'Н': 36, 'э': 37, 's': 38, 'n': 39, 'r': 40, 'i': 41, 'a': 42, 'П': 43, '-': 44, 'u': 45, 'o': 46, 't': 47, 'А': 48, 'В': 49, 'ф': 50, 'l': 51, '3': 52, ';': 53, 'О': 54, '!': 55, '?': 56, '2': 57, 'К': 58, 'Б': 59, 'm': 60, 'Д': 61, 'М': 62, 'd': 63, 'c': 64, 'С': 65, 'Р': 66, '&': 67, '#': 68, '…': 69, 'И': 70, 'p': 71, 'Т': 72, 'v': 73, ':': 74, 'Я': 75, '»': 76, '«': 77, '’': 78, '4': 79, 'h': 80, 'Г': 81, '1': 82, 'ъ': 83, '[': 84, ']': 85, 'Э': 86, 'Ч': 87, 'q': 88, 'Е': 89, 'f': 90, ')': 91, '(': 92, 'b': 93, 'g': 94, 'Л': 95, '5': 96, 'I': 97, '8': 98, '0': 99, 'З': 100, 'У': 101, '6': 102, '9': 103, '7': 104, 'z': 105, 'j': 106, 'Ж': 107, 'V': 108, 'M': 109, 'B': 

In [38]:
print(idx_to_char)

{0: ' ', 1: 'о', 2: 'а', 3: 'е', 4: 'и', 5: 'н', 6: 'т', 7: 'с', 8: 'л', 9: 'в', 10: 'р', 11: 'к', 12: 'д', 13: ',', 14: 'м', 15: 'у', 16: 'п', 17: 'я', 18: 'ь', 19: 'г', 20: 'ы', 21: 'з', 22: 'б', 23: 'ч', 24: '\n', 25: '.', 26: 'й', 27: 'ж', 28: 'ш', 29: 'х', 30: '–', 31: '\xa0', 32: 'ю', 33: 'e', 34: 'ц', 35: 'щ', 36: 'Н', 37: 'э', 38: 's', 39: 'n', 40: 'r', 41: 'i', 42: 'a', 43: 'П', 44: '-', 45: 'u', 46: 'o', 47: 't', 48: 'А', 49: 'В', 50: 'ф', 51: 'l', 52: '3', 53: ';', 54: 'О', 55: '!', 56: '?', 57: '2', 58: 'К', 59: 'Б', 60: 'm', 61: 'Д', 62: 'М', 63: 'd', 64: 'c', 65: 'С', 66: 'Р', 67: '&', 68: '#', 69: '…', 70: 'И', 71: 'p', 72: 'Т', 73: 'v', 74: ':', 75: 'Я', 76: '»', 77: '«', 78: '’', 79: '4', 80: 'h', 81: 'Г', 82: '1', 83: 'ъ', 84: '[', 85: ']', 86: 'Э', 87: 'Ч', 88: 'q', 89: 'Е', 90: 'f', 91: ')', 92: '(', 93: 'b', 94: 'g', 95: 'Л', 96: '5', 97: 'I', 98: '8', 99: '0', 100: 'З', 101: 'У', 102: '6', 103: '9', 104: '7', 105: 'z', 106: 'j', 107: 'Ж', 108: 'V', 109: 'M', 110: 

Видим, что уникальных символов в тексте не так уж и много.

Для работы с нейронной сетью необходимо создать батчи. В случае текстовой информации батчи будут создаваться из последовательности индексов (по сути это строки текста) и использоваться для обучения сети.

Будем генерировать сразу обучающую выборку (то, на чем будем учить сеть) и таргет для нее. Таргет (правильные ответы для нейросети) — это текст, сдвинутый на один символ вперед.

Размерность тензора батча: [BATCH_SIZE x SEQ_LEN x 1]

In [39]:
SEQ_LEN = 256
BATCH_SIZE = 16

def get_batch(sequence):
    trains = []
    targets = []
    for _ in range(BATCH_SIZE):
        batch_start = np.random.randint(0, len(sequence) - SEQ_LEN)
        chunk = sequence[batch_start: batch_start + SEQ_LEN]
        train = torch.LongTensor(chunk[:-1]).view(-1, 1)
        target = torch.LongTensor(chunk[1:]).view(-1, 1)
        trains.append(train)
        targets.append(target)
    return torch.stack(trains, dim=0), torch.stack(targets, dim=0)

Теперь напишем функцию, которая предсказывает текст с помощью нашей обученной нейросети. Это удобно сделать заранее, 
    чтобы смотреть, что генерирует сеть во время обучения.
Сеть предсказывает нам вероятности следующей буквы, и мы с помощью этих вероятностей достаем случайно по одной букве. 
    Если повторить операцию 1000 раз, получим текст из 1000 символов.

Параметр start_text нам нужен, чтобы было что-то, для чего предсказывать следующий символ. У нас этот символ по 
    умолчанию — пробел, и задача сети сначала — предсказать следующий символ после пробела. 
    Потом — следующий после этих 2-х символов и т.д.

Параметр temp — это уровень «случайности» генерируемого текста. Так называемая «температура» с отсылкой к понятию
     «энтропии». Если поставить высокое значение — вероятность каждой буквы будет почти одинакова и 
    текст превратится в случайную белиберду. Если поставить низкое значение — каждый раз будем предсказывать одно и то же и можем зациклиться на одной фразе.

In [40]:
def evaluate(model, char_to_idx, idx_to_char, start_text=' ', prediction_len=200, temp=0.3):
    hidden = model.init_hidden()
    idx_input = [char_to_idx[char] for char in start_text]
    train = torch.LongTensor(idx_input).view(-1, 1, 1).to(device)
    predicted_text = start_text
    
    _, hidden = model(train, hidden)
        
    inp = train[-1].view(-1, 1, 1)
    
    for i in range(prediction_len):
        output, hidden = model(inp.to(device), hidden)
        output_logits = output.cpu().data.view(-1)
        p_next = F.softmax(output_logits / temp, dim=-1).detach().cpu().data.numpy()        
        top_index = np.random.choice(len(char_to_idx), p=p_next)
        inp = torch.LongTensor([top_index]).view(-1, 1, 1).to(device)
        predicted_char = idx_to_char[top_index]
        predicted_text += predicted_char
    
    return predicted_text

И наконец наша нейросеть. Она работает так:

Превращаем каждый символ на входе сети в вектор (так называемный эмбеддинг).
    
Далее передаем эти векторы LSTM слою. У этого слоя есть особенность: он работает не независимо для каждого символа, а помнит, что к нему раньше приходило на вход. Притом, помнит не все: ненужное он умеет забывать. Такие слои называют рекуррентными и часто используют при работе с последовательностями.
    
Выходы из LSTM слоя пропускаем через Dropout. Этот слой «мешает» сети учиться, чтобы ей сложнее было выучить весть текст. В противном случае, если сеть "запомнит" весь текст, то она сможет генерировать только его, в данном случае переобучение будет выглядеть именно так.
    
Дальше отправляем выход из Dropout на линейный слой размерности словаря, чтобы на выходе получить столько чисел, сколько у нас символов в словаре. Потом мы этот вектор чисел будем превращать в «вероятности» каждого символа с помощью функции softmax.

In [41]:
class TextRNN(nn.Module):
    
    def __init__(self, input_size, hidden_size, embedding_size, n_layers=1):
        super(TextRNN, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size
        self.n_layers = n_layers

        self.encoder = nn.Embedding(self.input_size, self.embedding_size)
        self.lstm = nn.LSTM(self.embedding_size, self.hidden_size, self.n_layers)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.hidden_size, self.input_size)
        
    def forward(self, x, hidden):
        x = self.encoder(x).squeeze(2)
        out, (ht1, ct1) = self.lstm(x, hidden)
        out = self.dropout(out)
        x = self.fc(out)
        return x, (ht1, ct1)
    
    def init_hidden(self, batch_size=1):
        return (torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device),
               torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device))

Теперь создаем нейросеть и обучаем ее. LSTM блок принимает немного другой формат батча:

[SEQ_LEN x BATCH_SIZE x 1], поэтому делаем permute для тензоров train и target, чтобы поменять 0 и 1 размерность местами.

Параметры нейросети, путем изменения которых можно повысить качество ее работы:

    hidden_size — влияет на сложность сети. Стоит повышать для текстов большого размера. Если выставить большое значение для текста маленького размера, то сеть просто выучит весь текст и будет генерировать его же.
    n_layers — опять же, влияет на сложность сети. Грубо говоря, позволяет делать несколько LSTM слоев подряд просто меняя эту цифру.
    embedding_size — размер обучаемого эмбеддинга. Можно выставить в несколько раз меньше размера словаря (числа уникальных символов в тексте) или примерно такой же. Больше — нет смысла.
    
Дальше — стандартный для PyTorch цикл обучения нейросети: выбираем функцию потерь, оптимизатор и настраиваем расписание, по которому меняем шаг оптимизатора. В нашем случае снижаем шаг в 2 раза, если ошибка (loss) не падает 5 шагов подряд.

Если очень грубо, то здесь мы много раз подаем в нейросеть разные кусочки текста и учим ее делать все меньше и меньше ошибок, когда она предсказывает следующую букву по предыдущему тексту.

In [46]:
%%time
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
#model = TextRNN(input_size=len(idx_to_char), hidden_size=128, embedding_size=128, n_layers=2)
model = TextRNN(input_size=len(idx_to_char), hidden_size=128, embedding_size=64, n_layers=4)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, amsgrad=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    patience=5, 
    verbose=True, 
    factor=0.5
)

n_epochs = 5000 #50000
loss_avg = []

for epoch in range(n_epochs):
    model.train()
    train, target = get_batch(sequence)
    train = train.permute(1, 0, 2).to(device)
    target = target.permute(1, 0, 2).to(device)
    hidden = model.init_hidden(BATCH_SIZE)

    output, hidden = model(train, hidden)
    loss = criterion(output.permute(1, 2, 0), target.squeeze(-1).permute(1, 0))
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    loss_avg.append(loss.item())
    if len(loss_avg) >= 50:
        mean_loss = np.mean(loss_avg)
        print(f'Loss: {mean_loss}')
        scheduler.step(mean_loss)
        loss_avg = []
        model.eval()
        predicted_text = evaluate(model, char_to_idx, idx_to_char)
        print(predicted_text)

Loss: 3.5177550077438355
                  а о,       ои              мо ое и       ае         с     ае  еен     оооо  нн  нн  е е   нвон   о о       ео  о а      р   о   ео   а  н  е l е    оо а   оо       е а ао о      е не 
Loss: 3.4047984886169433
  о о    ои а     т      е  а  о    о е            т а    ет          о     о         к о      т            о    т        о   а    о  а          л    а               ио  в           в   о н и          
Loss: 3.4377097940444945
   о  е    и а  о   а о  и    но  оо  о а т  и о оо тоа          о  о л   ое    о    оо     о  о    , о п  о ло  сд ли   е     ео        с а к     о о   ао е     о  ло оии  оо   а е  е    тоо  е    о  
Loss: 3.4313578701019285
   е  е а   о             ие   ао                    яле          е          ср е     о   а  о     ноае о  о о       е        е    о   и         о      е       ,   л о        л  и   оаа            о   
Loss: 3.424369158744812
           е оон е ое  е     ооо о     а е     от   ноо  а        н 

Loss: 1.8346446466445923
 подобдивив собом и в лица, на него не зная приведил и слова, как был подомандал слушал он подомандал он с которой востордил своему и была на него по своим, и с востордил и сказал он и все под нему фра
Loss: 1.8161804389953613
 и в то в после него полторами и только все воспотоленным себе было и он старались поднять и от поддертивал и в после него в голову и в том и старой и с гостей и постоял с своему и в своем постоял в по
Loss: 1.8035174870491029
 в своем и старой под положения после простом, которой был на поставить проводил он обрасно делал постоль подомния поставлялся и обратился в него в коли полодал он в своим с которой не было постарого п
Loss: 1.800578966140747
 подолдав старали протельно и слушал и которые с ней в полковил в своей на подошел не вы своем князь Андрей принять на слушал под него не должен и отвесно не воспоглуно с нем подостаи на это поднял на 
Loss: 1.7991848158836365
 не во в своим в доминить проводить притрестие с нему надо в гостино

Loss: 1.6811728000640869
 стретельно должно столковый человек должны он не постоянной своих после подомнил на очень постолько комнату к после от нем с ней было восполечим старой он весьма со все молодым собой и комнате и не ви
Loss: 1.6788480019569396
 он валусь и столь с совершенно служение в какой привести, что он в своем с своему как все делались и совершенно в король с возворать дело делать своем себе с середине он все было все с своем все столь
Loss: 1.6750845551490783
 столько обратился была привести поставляло прошевлением он был не потом постоянной от от восполешивал в ней под нее по столько с подрашила он был от него по все был одного в делах как он все поднечное
Loss: 1.6698539757728577
 именном было со всем не видала на своему подание, но после составляющим проводилась на то всем после подъезжала с ней и женщина, поданном с нем доброса, который не старали положения просил свое стола 
Loss: 1.6584931063652038
 он в обратился он страшно было не довольно постоянным столько на в

In [ ]:
n_epochs = 50000
Loss: 1.4364623022079468
Wall time: 16min 49s

Итак, наша сеть обучилась. Давайте что-нибудь сгенерируем:

In [47]:
model.eval()

print(evaluate(
    model, 
    char_to_idx, 
    idx_to_char, 
    temp=0.3, 
    prediction_len=500, 
    start_text='. '
    )
)

. он сказал он и беспокойно поразилась и в том обеду на своей прислушалась с своей восположение и пожал в своем с уставил на это в своей просил в своем в своей коли своего не видал гостиной своего столько как бы восторженно в своем возвратился успел приводил просто себе не слезами и не старой после все все в том, как как будто он не после него приставляющим с ней полковой не в собой в гостиной себя на него войсками в первый простой на него в ней было подошел к положение и собой от ней своей он слы


Можно изменять параметры temp и start_text. 
С помощью start_text можно попробовать «задать тему/направление» для генерируемого текста. 
Желательно, ту, которая есть в тексте, на котором сеть училась.


In [48]:
model.eval()

print(evaluate(
    model, 
    char_to_idx, 
    idx_to_char, 
    temp=0.2, 
    prediction_len=50, 
    start_text='Дина'
    )
)

Дина после него в ней на колинной голову и поданием пр


In [49]:
model.eval()

print(evaluate(
    model, 
    char_to_idx, 
    idx_to_char, 
    temp=0.6, 
    prediction_len=100, 
    start_text='мошка '
    )
)

мошка груд, возвелине в красневшего и себя.
 
 – Да, подтлешивить разговаривали выставляет десты все испол
